# ARGUS Phase 6 - Cross-fit Out-Of-Fold (OOF) SFT Training
Recipe: 01_sft.ipynb exact recipe (Qwen3.5-4B, 3 epochs, lr 2e-4, seed 3407, LoRA r=16, alpha=16, adamw_8bit).
Select FOLD (0..4) or run all folds. Takes final epoch 3 checkpoint (no checkpoint selection on validation).
Generates greedy out-of-fold predictions and exports GGUF (q4_k_m) for Ollama serve.

In [ ]:
# Official Unsloth Colab installation (reliable, no silent uv failures)
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!pip install transformers==5.2.0
!unsloth install-kernels || true


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/argus/data'
OUT_DIR  = '/content/drive/MyDrive/argus/out_crossfit'
BASE_MODEL = 'unsloth/Qwen3.5-4B'
FOLDS_TO_RUN = [0, 1, 2, 3, 4]  # Or [0, 1, 2] for 3-fold fallback
EPOCHS = 3
LR = 2e-4
SEED = 3407
import os, json, gc, torch
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq
from trl import SFTTrainer, SFTConfig

def load_model(name, max_len):
    try:
        m, t = FastLanguageModel.from_pretrained(name, max_seq_length=max_len, load_in_4bit=False, use_gradient_checkpointing='unsloth')
    except Exception as e:
        from unsloth import FastVisionModel
        m, t = FastVisionModel.from_pretrained(name, load_in_4bit=False, use_gradient_checkpointing='unsloth')
    return m, t

def add_lora(m):
    return FastLanguageModel.get_peft_model(m, r=16, lora_alpha=16, lora_dropout=0, bias='none', random_state=SEED,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], use_gradient_checkpointing='unsloth')

def text_tok(t):
    return getattr(t, 'tokenizer', t)

def prompt_text(tok, row):
    msgs = [{'role':'system','content':row['system']},{'role':'user','content':row['user']}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

def encode(tok, row, max_len):
    p = tok(prompt_text(tok, row), add_special_tokens=False)['input_ids']
    c = tok(row['assistant'] + '<|im_end|>', add_special_tokens=False)['input_ids']
    ids = (p + c)[:max_len]
    labels = ([-100] * len(p) + c)[:max_len]
    return {'input_ids': ids, 'attention_mask': [1] * len(ids), 'labels': labels}

@torch.no_grad()
def generate_greedy(model, tok, rows, max_new=300, batch=16):
    tok.padding_side = 'left'
    outs = []
    for i in range(0, len(rows), batch):
        chunk = rows[i:i+batch]
        enc = tok([prompt_text(tok, r) for r in chunk], return_tensors='pt', padding=True, add_special_tokens=False).to('cuda')
        gen = model.generate(**enc, max_new_tokens=max_new, do_sample=False, use_cache=True)
        new = gen[:, enc['input_ids'].shape[1]:]
        texts = tok.batch_decode(new, skip_special_tokens=True)
        for j in range(len(chunk)):
            outs.append(texts[j].strip())
    return outs

In [ ]:
# Train loop over requested folds
for fold in FOLDS_TO_RUN:
    print(f'\n========== STARTING FOLD {fold} ==========')
    fold_train_path = f'{DATA_DIR}/sft_train_fold{fold}.jsonl'
    with open(fold_train_path, encoding='utf-8') as fh:
        train_rows = [json.loads(line) for line in fh if line.strip()]
    print(f'Fold {fold} loaded {len(train_rows)} training rows')
    
    # Context length calculation
    model, tokenizer = load_model(BASE_MODEL, 1024)
    tok = text_tok(tokenizer)
    lens = sorted(len(tok(prompt_text(tok, r), add_special_tokens=False)['input_ids']) + len(tok(r['assistant'], add_special_tokens=False)['input_ids']) + 2 for r in train_rows)
    max_len = max(1024, ((lens[-1] + 63) // 64) * 64)
    print(f'Fold {fold}: MAX_LEN = {max_len}')
    
    model = add_lora(model)
    ds = Dataset.from_list([encode(tok, r, max_len) for r in train_rows])
    fold_out = f'{OUT_DIR}/fold{fold}'
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=ds,
        data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100),
        args=SFTConfig(
            per_device_train_batch_size=8, gradient_accumulation_steps=2, warmup_steps=5,
            num_train_epochs=EPOCHS, learning_rate=LR, logging_steps=5, optim='adamw_8bit', weight_decay=0.001,
            lr_scheduler_type='linear', seed=SEED, output_dir=fold_out, report_to='none',
            save_strategy='epoch', save_total_limit=None,
            remove_unused_columns=False, dataset_text_field='', dataset_kwargs={'skip_prepare_dataset': True}, max_length=max_len))
    trainer.train()
    
    # Final epoch checkpoint is checkpoint-3 (no checkpoint selection on validation)
    import glob
    ckpts = sorted(glob.glob(f'{fold_out}/checkpoint-*'), key=lambda p: int(p.rsplit('-', 1)[1]))
    final_ckpt = ckpts[-1]
    print(f'Fold {fold}: final epoch-3 checkpoint is {final_ckpt}')
    
    # Free training memory before generation and GGUF conversion
    del model, trainer, ds; gc.collect(); torch.cuda.empty_cache()
    
    # Evaluate out-of-fold excluded keys for greedy pass@1
    m, t = load_model(final_ckpt, 2048)
    FastLanguageModel.for_inference(m)
    tk = text_tok(t)
    
    # Load all sft_train to get excluded fold rows
    with open(f'{DATA_DIR}/sft_train.jsonl', encoding='utf-8') as fh:
        all_rows = [json.loads(line) for line in fh if line.strip()]
    fold_keys = set(r['input_key'] for r in train_rows)
    oof_eval_rows = [r for r in all_rows if r['input_key'] not in fold_keys]
    # Deduplicate by input_key for distinct cluster evaluation
    distinct_oof = []
    seen_keys = set()
    for r in oof_eval_rows:
        if r['input_key'] not in seen_keys:
            seen_keys.add(r['input_key'])
            distinct_oof.append(r)
    
    preds = generate_greedy(m, tk, distinct_oof)
    pred_out_path = f'{OUT_DIR}/fold{fold}_preds.jsonl'
    with open(pred_out_path, 'w', encoding='utf-8') as fh:
        for r, pred in zip(distinct_oof, preds):
            fh.write(json.dumps({'input_key': r['input_key'], 'content': pred}) + '\n')
    print(f'Fold {fold}: wrote {len(distinct_oof)} greedy OOF predictions to {pred_out_path}')
    
    # Export GGUF
    gguf_dir = f'{OUT_DIR}/fold{fold}_gguf'
    m.save_pretrained_gguf(gguf_dir, t, quantization_method='q4_k_m')
    print(f'Fold {fold}: exported GGUF to {gguf_dir}')
    
    # Write Modelfile for Ollama
    modelfile_content = (f'FROM ./fold{fold}/argus-m1-sft-fold{fold}.Q4_K_M.gguf\n'
                         f'PARAMETER presence_penalty 1.5\n'
                         f'PARAMETER top_k 20\n'
                         f'PARAMETER top_p 0.95\n'
                         f'PARAMETER num_predict 1024\n')
    with open(f'{OUT_DIR}/Modelfile.fold{fold}', 'w') as fh:
        fh.write(modelfile_content)
    
    del m, t; gc.collect(); torch.cuda.empty_cache()
    print(f'========== FOLD {fold} COMPLETE ==========')


In [ ]:
# Cell 5: Compute per-fold greedy pass@1 on excluded fold clusters using Phase 5 gate scorer
import os
from ml.train_m1.score_oof import score_oof_file
from ml.mutation_engine.dataset_io import input_key, read_jsonl

train_dataset = read_jsonl(f'{DATA_DIR}/train.jsonl')
train_items = {}
train_counts = {}
for ex in train_dataset:
    k = input_key(ex)
    train_items.setdefault(k, ex)
    train_counts[k] = train_counts.get(k, 0) + 1

print('\n' + '='*75)
print(f"{'Fold':<6} | {'Excluded Clusters':<18} | {'Gate Pass@1':<25} | {'Schema Valid':<12}")
print('='*75)
for fold in FOLDS_TO_RUN:
    pred_path = f'{OUT_DIR}/fold{fold}_preds.jsonl'
    if os.path.exists(pred_path):
        res = score_oof_file(pred_path, train_items, train_counts)
        print(f"Fold {fold:<2} | {res['keys_matched']:<18} | {res['pass_at_1']*100:.1f}% ({res['weighted_pass']}/{res['total_examples']}) | {res['valid_schema']}/{res['keys_matched']}")
print('='*75)
